# Resolution Forecasting Baselines

This notebook profiles the exported resolution dataset and inspects the rolling backtest that compares the market-implied YES probability against ChaosWing's expanding-window logistic challenger.

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")

DATA_DIR = Path("../ml_data") if Path("../ml_data").exists() else Path("ml_data")

def load_jsonl(path: Path) -> pd.DataFrame:
    if not path.exists():
        return pd.DataFrame()
    with path.open("r", encoding="utf-8") as handle:
        rows = [json.loads(line) for line in handle if line.strip()]
    return pd.DataFrame(rows)

forecast_df = load_jsonl(DATA_DIR / "resolution_forecast_examples.jsonl")
experiments_df = load_jsonl(DATA_DIR / "experiments.jsonl")

forecast_df.head()

In [ ]:
dataset_profile = {
    "examples": len(forecast_df),
    "events": forecast_df.get("event_slug", pd.Series(dtype=str)).nunique() if not forecast_df.empty else 0,
    "positive_rate": forecast_df.get("target", pd.Series(dtype=float)).mean() if "target" in forecast_df else None,
}
dataset_profile


In [ ]:
resolution_runs = experiments_df[experiments_df.get("task_type", pd.Series(dtype=str)) == "resolution_backtest"].copy()
resolution_runs[["title", "dataset_version", "metrics"]].tail()


In [ ]:
if not resolution_runs.empty:
    latest = resolution_runs.iloc[-1]
    evaluated = pd.DataFrame((latest.get("artifacts") or {}).get("evaluated_examples") or [])
    if not evaluated.empty:
        bins = pd.cut(evaluated["model_probability"], bins=[0, 0.2, 0.4, 0.6, 0.8, 1.0], include_lowest=True)
        calibration = evaluated.groupby(bins, observed=False).agg(
            observed_rate=("target", "mean"),
            avg_prediction=("model_probability", "mean"),
            count=("target", "size"),
        ).reset_index()
        display(calibration)
        plt.figure(figsize=(6, 4))
        plt.plot([0, 1], [0, 1], linestyle="--", color="black", label="Perfect calibration")
        plt.plot(calibration["avg_prediction"], calibration["observed_rate"], marker="o", label="Model")
        plt.title("Resolution Forecast Calibration")
        plt.xlabel("Average predicted probability")
        plt.ylabel("Observed positive rate")
        plt.legend()
        plt.show()
    else:
        print("No evaluated_examples artifacts were found in the latest resolution backtest run.")
else:
    print("No resolution_backtest experiment rows were found.")


## Findings

- Summarize whether the logistic challenger improves on the market-implied baseline.
- Call out which feature slices or market regimes look strongest or weakest.

## Limitations

- Note current sample-size limits and any class-balance issues.
- Record whether calibration analysis is constrained by missing experiment artifacts.